In [22]:
# Cell 1: Setup
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')
from src.simulation import run_simulation
from src.myth_writer import MythWriter
from games.trust_game import TrustGame


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
# Cell 2: Quick 10-round test
game = TrustGame(endowment=5, multiplier=3)
myth_writer = MythWriter(myth_topic="")
model = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo"

# All possible task orders
all_task_orders = [
    ["game"],
    ["myth"],
    ["game", "myth"],
    ["myth", "game"]
]

base_path = "../data/json/main_loop/task_order/"

# Run simulation for each task order
for task_order in all_task_orders:
    print(f"\n{'='*80}")
    print(f"Running simulation with task_order: {task_order}")
    print(f"{'='*80}\n")
    
    sim_data = run_simulation(
        game, model,
        temperature=0.8,
        num_turns=20,  # Short for testing
        num_agents=2,
        memory_capacity=6,
        agent_biases="",
        myth_writer=myth_writer,
        task_order=task_order
    )
    
    # Create filename based on task_order
    task_order_str = "_".join(sim_data.task_order)
    filename = f"{task_order_str}.json"
    
    # Save state
    sim_data.save_state(base_path + filename)
    print(f"\n✓ Simulation state saved to {filename}")


Running simulation with task_order: ['game']

SIMULATION: TrustGame
Number of rounds: 20
Number of agents: 2
Memory capacity: 6
Role swapping: ENABLED (agents alternate roles each round)
--------------------------------------------------------------------------------

ROUND 1
Roles this round: Agent_1 = INVESTOR, Agent_2 = TRUSTEE

--- PHASE 1: GAME PLAY ---

Agent_1 (Investor) prompt: Round 1: You are the INVESTOR. You have $5. How much do you send? (0-5)
Agent_1 (Investor) response: {'send': 3, 'reason': 'I will start by sending a moderate amount, showing my willingness to cooperate and seeing how the Trustee responds.'}

Agent_2 (Trustee) prompt: Round 1: You are the TRUSTEE. Investor sent $3.0, that is 60.0% of its total endowment. You received $9.0. How much do you return? (0-9.0)
Agent_2 (Trustee) response: {'return': 4.5, 'reason': 'I will return half of the received amount to reciprocate the investor's trust. This will also ensure a fair distribution and encourage cooperation 

KeyboardInterrupt: 

### Analyses


In [ ]:
# Cell 3: Run analyses for each task order
import subprocess
import os


# All possible task orders
all_task_orders = [
    ["game"],
    ["myth"],
    ["game", "myth"],
    ["myth", "game"]
]

# Run analyses for each task order (using the same all_task_orders from above)
for task_order in all_task_orders:
    task_order_str = "_".join(task_order)

    print(f"\n{'='*80}")
    print(f"Running analyses for task_order: {task_order}")
    print(f"{'='*80}")

    # Run shell script with task order argument - stream output in real-time
    process = subprocess.Popen([
        "bash",
        "../scripts/run_analyses_clean.sh",
        task_order_str
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True)

    # Print output line by line as it comes
    for line in process.stdout:
        print(line, end='')

    # Wait for process to complete
    process.wait()

    print(f"\n✓ Analyses completed for {task_order_str}")

print(f"\n{'='*80}")
print("All analyses completed!")
print(f"{'='*80}")


Running analyses for task_order: ['game', 'myth']
Running analyses for task order: game_myth
Input file: /Users/ivar/Desktop/current_projects/AI_projects/test_modular_llm/data/json/main_loop/task_order/game_myth.json
Output directory: /Users/ivar/Desktop/current_projects/AI_projects/test_modular_llm/data/plots/task_order/game_myth

Running cooperativity_analysis.py...
Reading myths from /Users/ivar/Desktop/current_projects/AI_projects/test_modular_llm/data/json/main_loop/task_order/game_myth.json...
Found 2 agents
  Agent_1: 20 rounds
  Agent_2: 20 rounds

Analyzing cooperativity...

COOPERATIVITY ANALYSIS

────────────────────────────────────────────────────────────────────────────────
AGENT: Agent_1
────────────────────────────────────────────────────────────────────────────────

Baseline (Round 1):
  Mutuality Ratio: 4.500
  Cooperative Language: 8.65%
  Uncooperative Language: 0.96%

Average across all rounds:
  Mutuality Ratio: 14.225
  Cooperative Language: 12.86%
  Uncooperativ

In [ ]:
# Cell 3: Parameter sweep
temperatures = [0.1,0.7, 1.0]
results = {}

for temp in temperatures:
    print(f"\n{'='*80}\nTesting temperature={temp}\n{'='*80}")
    game = TrustGame(endowment=5, multiplier=3)
    myth_writer = MythWriter(myth_topic="")
    
    sim_data = run_simulation(
        game, model,
        temperature=temp,
        num_turns=20,
        num_agents=2,
        memory_capacity=3,
        agent_biases="",
        myth_writer=myth_writer
    )
    results[temp] = sim_data

In [ ]:
# Cell 4: Quick analysis
import matplotlib.pyplot as plt
import numpy as np

colors = plt.cm.tab10(range(len(results)))  # Get distinct colors for each temperature

for idx, (temp, sim_data) in enumerate(results.items()):
    sent_amounts = [r['sent'] for r in sim_data.conversation_history]
    returned_amounts = [r['returned'] for r in sim_data.conversation_history]
    color = colors[idx]
    plt.plot(sent_amounts, label=f'temp={temp} (sent)', linestyle='-', linewidth=2, color=color)
    plt.plot(returned_amounts, label=f'temp={temp} (returned)', linestyle=':', linewidth=2, color=color)

plt.xlabel('Round')
plt.ylabel('Amount')
plt.xticks(range(len(sent_amounts)))  # Show only integer round numbers
plt.legend()
plt.title('Trust Evolution by Temperature (Sent vs Returned)')
plt.show()